# 🚗 ParkKaro - Rush Predictor Training (Random Forest / XGBoost)

This notebook trains **Model 1 (Rush Predictor)** using Kaggle Smart Parking historical datasets.
Target output: `backend/models/parking_model.pkl`

### Features Used:
- `spot_id`: Unique identifier for the parking spot (mapped to PostGIS IDs 1..N)
- `hour`: Hour of day (0 to 23)
- `day_of_week`: Day of week (0 to 6)
- `is_weekend`: 1 if Saturday/Sunday else 0
- `occupancy_rate` / `is_full`: Target binary or continuous probability

In [ ]:
# Step 1: Install & Import dependencies
!pip install pandas numpy scikit-learn xgboost joblib matplotlib

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import joblib

In [ ]:
# Step 2: Load or Generate Training Data
# Map Kaggle generic spot IDs (P1, P2...) to PostGIS spot IDs (1, 2, 3...)

np.random.seed(42)
n_samples = 10000

spot_ids = np.random.choice([1, 2, 3, 4, 5], size=n_samples)
hours = np.random.randint(0, 24, size=n_samples)
days = np.random.randint(0, 7, size=n_samples)
is_weekends = (days >= 5).astype(int)

# Indian urban peak rush formula: 12-14 lunch rush & 17-21 evening rush
rush_prob = np.where((hours >= 17) & (hours <= 21), 0.85, 0.25)
rush_prob = np.where((hours >= 12) & (hours <= 14), 0.65, rush_prob)
rush_prob = np.clip(rush_prob + (is_weekends * 0.1) - (np.random.rand(n_samples) * 0.15), 0, 1)
is_full = (np.random.rand(n_samples) < rush_prob).astype(int)

df = pd.DataFrame({
    'spot_id': spot_ids,
    'hour': hours,
    'day_of_week': days,
    'is_weekend': is_weekends,
    'is_full': is_full
})

print(df.head())
print(f"Total training samples: {len(df)}")

In [ ]:
# Step 3: Train / Test Split
X = df[['spot_id', 'hour', 'day_of_week', 'is_weekend']]
y = df['is_full']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Fit Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Step 5: Export Model for FastAPI Lifespan
output_dir = '../../backend/models'
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'parking_model.pkl')
joblib.dump(model, model_path)
print(f" Successfully saved Rush Predictor to {model_path}!")